In [1]:
# ── Imports ──────────────────────────────────────────────
import sqlite3
import pandas as pd

## Step 1 — Database Connection & Table Creation

In [2]:
def get_connection(db_name: str = "Sales_analysis.db") -> sqlite3.Connection:
    """Create and return a database connection."""
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

conn = get_connection()
cursor = conn.cursor()
print(f'Database Created Successfully')

Database Created Successfully


In [3]:
def create_tables(cursor: sqlite3.Cursor) -> None:
    """Create all tables with IF NOT EXISTS to avoid duplicate errors."""

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS customers (
           customer_id   INT           PRIMARY KEY, 
           first_name    VARCHAR(50)   NOT NULL, 
           last_name     VARCHAR(50)   NOT NULL, 
           email         VARCHAR(100)  UNIQUE NOT NULL, 
           city          VARCHAR(50)   NOT NULL, 
           state         VARCHAR(50)   NOT NULL, 
           join_date     DATE          NOT NULL, 
           is_premium    BOOLEAN       DEFAULT FALSE
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS products (
            product_id    INT           PRIMARY KEY, 
            product_name  VARCHAR(100)  NOT NULL, 
            category      VARCHAR(50)   NOT NULL, 
            brand         VARCHAR(50)   NOT NULL, 
            unit_price    DECIMAL(10,2) NOT NULL  CHECK (unit_price > 0), 
           stock_qty     INT           NOT NULL  DEFAULT 0  CHECK (stock_qty >= 0) 

        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS orders (
            order_id      INT           PRIMARY KEY, 
            customer_id   INT           NOT NULL, 
            order_date    DATE          NOT NULL, 
            status        VARCHAR(20)   NOT NULL  DEFAULT 'Pending' 
                  CHECK (status IN ('Pending','Shipped','Delivered','Cancelled')), 
            total_amount  DECIMAL(12,2) NOT NULL  CHECK (total_amount >= 0), 
     
            FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS order_items (
             item_id       INT           PRIMARY KEY, 
             order_id      INT           NOT NULL, 
             product_id    INT           NOT NULL, 
             quantity      INT           NOT NULL  CHECK (quantity > 0), 
             unit_price    DECIMAL(10,2) NOT NULL  CHECK (unit_price > 0), 
             discount_pct  DECIMAL(5,2)  DEFAULT 0 CHECK (discount_pct BETWEEN 0 AND 100), 
             FOREIGN KEY (order_id)   REFERENCES orders(order_id), 
             FOREIGN KEY (product_id) REFERENCES products(product_id)
            ) 
    """)

    conn.commit()
    print("All tables created successfully")

create_tables(cursor)

All tables created successfully


In [4]:
def create_indexes(cursor: sqlite3.Cursor) -> None:
    """Create indexes for faster query performance."""
    indexes = [
        "CREATE INDEX IF NOT EXISTS idx_customers_city  ON customers(city)",
        "CREATE INDEX IF NOT EXISTS idx_customers_state ON customers(state)",
        "CREATE INDEX IF NOT EXISTS  idx_products_category ON products(category)",
        "CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id)",
        "CREATE INDEX IF NOT EXISTS idx_items_order ON order_items(order_id)",
    ]
    for idx in indexes:
        cursor.execute(idx)
    conn.commit()
    

create_indexes(cursor)
print("Indexes Created")

Indexes Created


## Step 2 — Insert Sample Data

In [5]:
def insert_customers(cursor: sqlite3.Cursor) -> None:
    
    customers = [
        (101, 'Aarav',  'Sharma', 'aarav.s@email.com',  'Mumbai',    'Maharashtra', '2024-01-15', 'TRUE'),
        (102 ,'Priya', 'Patel' ,'priya.p@email.com' ,'Ahmedabad' ,'Gujarat', '2024-02-20', 'FALSE'),
        (103 ,'Rohan', 'Gupta','rohan.g@email.com' ,'Delhi' ,'Delhi ','2024-03-10','TRUE'),
        (104 ,'Sneha', 'Reddy' ,'sneha.r@email.com','Hyderabad' ,'Telangana','2024-04-05' ,'FALSE'),
        (105, 'Vikram' ,'Singh', 'vikram.s@email.com', 'Jaipur','Rajasthan', '2024-05-12' ,'TRUE'), 
        (106,'Ananya', 'Iyer','ananya.i@email.com' ,'Chennai','Tamil Nadu' ,'2024-06-18' ,'FALSE'),
        (107,'Karan', 'Mehta','karan.m@email.com', 'Pune', 'Maharashtra', '2024-07-22', 'TRUE'),
        (108, 'Divya','Nair', 'divya.n@email.com', 'Kochi','Kerala', '2024-08-30', 'FALSE')

]
    cursor.executemany("""
        INSERT OR IGNORE INTO customers
        VALUES (?,?,?,?,?,?,?,?)
    """, customers)
    conn.commit()

insert_customers(cursor)

In [6]:
def insert_products(cursor: sqlite3.Cursor) -> None:
    """Insert product records."""
    products = [
        (201, 'Wireless Earbuds', 'Electronics','BoAt', 1499.0, 250),
        (202,'Cotton T-Shirt' ,'Clothing' ,'Levis' ,799.0 ,500),
        (203 ,'Smart Watch' ,'Electronics','Noise',2999.0 ,150),
        (204 ,'Running Shoes' ,'Clothing','Nike' ,4599.0 ,120),
        (205,'Bluetooth Speaker','Electronics','JBL',3499.0 ,200),
        (206 ,'Bedsheet Set' ,'Home','Spaces' ,1299.0 ,300),
        (207 ,'Laptop Stand' ,'Electronics','AmazonBasics' ,899.0 ,180),
        (208 ,'Cushion Covers(Set)' ,'Home' ,'HomeCenter' ,599.0 ,400)
    ]
    cursor.executemany("""
        INSERT OR IGNORE INTO products
        VALUES (?,?,?,?,?,?)
    """, products)
    conn.commit()

insert_products(cursor)

In [7]:
def insert_orders_and_items(cursor: sqlite3.Cursor) -> None:
    """Insert orders and order line items."""
    orders = [
        (1001, 101, '2024-08-01', 'Delivered',4498),
        (1002 ,102 ,'2024-08-03' ,'Delivered', 799.0),
        (1003 ,103 ,'2024-08-05' ,'Shipped' ,7498.0),
        (1004 ,101 ,'2024-08-10' ,'Delivered' ,3499.0),
        (1005 ,104 ,'2024-08-12' ,'Cancelled' ,2999.0),
        (1006 ,105 ,'2024-08-15' ,'Delivered' ,5898.0),
        (1007 ,106 ,'2024-08-18', 'Pending' ,1299.0),
        (1008 ,103 ,'2024-08-20' ,'Delivered',899.0) ,
        (1009 ,107 ,'2024-08-25' ,'Shipped' ,6098.0),
        (1010, 108 ,'2024-08-28' ,'Delivered' ,1598.0) 


    ]
    cursor.executemany("INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?)", orders)

    order_items = [
        (5001,1001, 201, 2, 1499.0,0),
        (5002,1001, 207, 1, 899.0,10),
        (5003 ,1002 ,202 ,1 ,799.0 ,0),
        (5004 ,1003 ,203 ,1 ,2999.0 ,0),
        (5005 ,1003 ,204 ,1 ,4599.0 ,5),
        (5006 ,1004 ,205 ,1 ,3499.0 ,0),
        (5007 ,1005 ,203 ,1 ,2999.0 ,0),
        (5008 ,1006 ,201 ,1 ,1499.0 ,10), 
        (5009 ,1006 ,204 ,1 ,4599.0 ,5),
        (5010 ,1007 ,206 ,1 ,1299.0 ,0),
        (5011 ,1008 ,207 ,1 ,899.0 ,0 ),
        (5012 ,1009 ,205 ,1 ,3499.0 ,0 ),
        (5013 ,1009 ,208 ,2 ,599.0 ,15 ),
        (5014 ,1010 ,206 ,1 ,1299.0 ,0),
        (5015 ,1010 ,208 ,1 ,599.0 ,0)


    ]
    cursor.executemany("INSERT OR IGNORE INTO order_items VALUES (?,?,?,?,?,?)", order_items)
    conn.commit()

insert_orders_and_items(cursor)
print("All values are inserted")

All values are inserted


## Section A — SQL Basics (SELECT, Constraints, Primary Keys) 

Q1. Write a query to display all columns and rows from the customer's table. 

In [8]:
pd.read_sql("SELECT * FROM customers", conn)

,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,TRUE
1,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,FALSE
2,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,TRUE
3,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,FALSE
4,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,TRUE
5,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,FALSE
6,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,TRUE
7,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,FALSE


Q2. Retrieve only the first_name, last_name, and city of all customers. 

In [9]:
pd.read_sql("Select first_name,last_name,city from customers",conn)

,first_name,last_name,city
0,Aarav,Sharma,Mumbai
1,Priya,Patel,Ahmedabad
2,Rohan,Gupta,Delhi
3,Sneha,Reddy,Hyderabad
4,Vikram,Singh,Jaipur
5,Ananya,Iyer,Chennai
6,Karan,Mehta,Pune
7,Divya,Nair,Kochi


Q3. List all unique categories available in the products table. 

In [10]:
pd.read_sql("select distinct category from products",conn)

,category
0,Clothing
1,Electronics
2,Home


Q4. Identify the Primary Key of each table in the schema. Explain why a Primary Key must be unique and NOT NULL. 

Identifying Primary key in each table

Customers Table
Primary Key: customer_id
Reason: It uniquely identifies each customer in the dataset.
Orders Table
Primary Key: order_id
Reason: Each order placed is unique and cannot be duplicated.
Products Table
Primary Key: product_id
Reason: Each product has a unique identifier that distinguishes it from others.
Sales Table
Primary Key: transaction_id (or order_id + product_id if composite key is used)
Reason: Each sales record represents a unique transaction.


Primary Key must be unique and NOT NUll because:-
It prevents duplicate records
Ensures each row represents a real-world unique entity
Helps in JOIN operations between tables
Without a value uniqueness is not meaningful


Q5. What constraints are applied to the email column in the customers table? What would happen if you tried to insert a duplicate email? 

The UNIQUE , NOT NULL constarint is applied to the email column in customers table .If we tried to insert a duplicate email,the database will reject the insert operation and throw a constraint violation error (typically a UNIQUE constraint violation).


Q6. Try inserting a product with unit_price = -50. What happens and which constraint prevents it? Write both the INSERT statement and explain the error. 

In [11]:

try:
    cursor.execute("""
        INSERT INTO products VALUES(209,'Bedsheets','Home','HomeCenter',-50,400)
    """)
except Exception as e:
    print(f" Error: {e}")

#The CHECK constarint is preventing the statement to execute
#The error "CHECK constraint failed: unit_price >0" occurs as the CHECK constraint is applied in the unit_price price .The statement will work only when the unit_price>0

 Error: CHECK constraint failed: unit_price > 0


##SECTION B (Filtering & Optimization)

Q7. Retrieve all orders with status = 'Delivered'. 

In [12]:
pd.read_sql("Select * from orders where status ='Delivered'",conn)


,order_id,customer_id,order_date,status,total_amount
0,1001,101,2024-08-01,Delivered,4498
1,1002,102,2024-08-03,Delivered,799
2,1004,101,2024-08-10,Delivered,3499
3,1006,105,2024-08-15,Delivered,5898
4,1008,103,2024-08-20,Delivered,899
5,1010,108,2024-08-28,Delivered,1598


Q8. Find all products in the 'Electronics' category with a unit_price greater than ₹2000. 

In [13]:
pd.read_sql("select * from products where category='Electronics' AND unit_price>2000",conn)


,product_id,product_name,category,brand,unit_price,stock_qty
0,203,Smart Watch,Electronics,Noise,2999,150
1,205,Bluetooth Speaker,Electronics,JBL,3499,200


Q9. List all customers who joined in the year 2024 and belong to the state 'Maharashtra'. 

In [14]:
pd.read_sql("select * from customers WHERE join_date >= '2024-01-01' AND join_date < '2025-01-01'AND state='Maharashtra'",conn)


,customer_id,first_name,last_name,email,city,state,join_date,is_premium
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,TRUE
1,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,TRUE


Q10. Find all orders placed between '2024-08-10' and '2024-08-25' (inclusive) that are NOT cancelled. 

In [15]:
pd.read_sql("Select * from orders where order_date BETWEEN '2024-08-10' AND '2024-08-25' AND status <> 'Cancelled'",conn)
               

,order_id,customer_id,order_date,status,total_amount
0,1004,101,2024-08-10,Delivered,3499
1,1006,105,2024-08-15,Delivered,5898
2,1007,106,2024-08-18,Pending,1299
3,1008,103,2024-08-20,Delivered,899
4,1009,107,2024-08-25,Shipped,6098


Q11. Explain what the index idx_orders_date does. How would it improve the performance of a query that filters orders by order_date? Write a sample query that would benefit from this index. 

The index idx_orders_date is an index created on the order_date column of the orders table.
An index stores the values of the indexed column in a sorted structure, allowing the database to locate matching rows quickly without scanning the entire table.This significantly speeds up searches, especially when the orders table contains a large number of records.
Query that benefits from this index:
SELECT *
FROM orders
WHERE order_date = '2024-08-15';

Q12. If you run: SELECT * FROM customers WHERE YEAR(join_date) = 2024; — would the index on join_date be used? Explain why or why not, and rewrite the query to be index-friendly (SARGable). 

No, the index on join_date would typically not be used efficiently on this query.The column is wrapped inside a function(YEAR), the database generally cannot use the index to directly locate matching rows. Instead, it may need to scan all rows and calculate YEAR(join_date) for each one.
Index friendly query:
SELECT *
FROM customers
WHERE join_date >= '2024-01-01'
  AND join_date < '2025-01-01';


##Section C — Aggregation (GROUP BY, SUM, COUNT, AVG, MIN, MAX) 

Q13. Count the total number of orders in the orders table. 

In [16]:
pd.read_sql("Select COUNT(order_id) from orders",conn)


,COUNT(order_id)
0,10


Q14. Find the total revenue (SUM of total_amount) from all 'Delivered' orders. 

In [17]:
pd.read_sql("Select SUM(total_amount) from orders where status='Delivered'",conn)


,SUM(total_amount)
0,17191


Q15. Calculate the average unit_price of products in each category. 

In [18]:
pd.read_sql("Select AVG(unit_price) as avg_price from products group by category",conn)


,avg_price
0,2699.0
1,2224.0
2,949.0


Q16. For each order status, find the count of orders and the total revenue. Sort the result by total revenue in descending order. 

In [19]:
pd.read_sql("Select COUNT(order_id),SUM(total_amount) from orders group by status order by total_amount DESC",conn)


,COUNT(order_id),SUM(total_amount)
0,2,13596
1,6,17191
2,1,2999
3,1,1299


Q17. Find the most expensive (MAX) and cheapest (MIN) product in each category. 

In [20]:
pd.read_sql("Select MAX(unit_price),MIN(unit_price) from products group by category",conn)


,MAX(unit_price),MIN(unit_price)
0,4599,799
1,3499,899
2,1299,599


Q18. List all product categories where the average unit_price is greater than ₹2000. (Hint: Use HAVING clause) 

In [21]:
pd.read_sql("Select * from products group by category Having avg(unit_price) >2000",conn)


,product_id,product_name,category,brand,unit_price,stock_qty
0,202,Cotton T-Shirt,Clothing,Levis,799,500
1,201,Wireless Earbuds,Electronics,BoAt,1499,250


##Section D — Joins & Relationships 

Q19. Write an INNER JOIN query to display each order along with the customer's first_name and last_name. Show: order_id, order_date, first_name, last_name, total_amount. 

In [22]:
pd.read_sql("Select c.first_name,c.last_name,order_id,order_date,total_amount from customers c INNER JOIN orders o on c.customer_id =o.customer_id",conn)


,first_name,last_name,order_id,order_date,total_amount
0,Aarav,Sharma,1001,2024-08-01,4498
1,Priya,Patel,1002,2024-08-03,799
2,Rohan,Gupta,1003,2024-08-05,7498
3,Aarav,Sharma,1004,2024-08-10,3499
4,Sneha,Reddy,1005,2024-08-12,2999
5,Vikram,Singh,1006,2024-08-15,5898
6,Ananya,Iyer,1007,2024-08-18,1299
7,Rohan,Gupta,1008,2024-08-20,899
8,Karan,Mehta,1009,2024-08-25,6098
9,Divya,Nair,1010,2024-08-28,1598


Q20. Using a LEFT JOIN, list ALL customers and their orders (if any). Customers with no orders should still appear with NULL values for order columns. 

In [23]:
pd.read_sql("Select * from customers c LEFT JOIN orders o on c.customer_id=o.customer_id",conn)


,customer_id,first_name,last_name,email,city,state,join_date,is_premium,order_id,customer_id,order_date,status,total_amount
0,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,TRUE,1001,101,2024-08-01,Delivered,4498
1,101,Aarav,Sharma,aarav.s@email.com,Mumbai,Maharashtra,2024-01-15,TRUE,1004,101,2024-08-10,Delivered,3499
2,102,Priya,Patel,priya.p@email.com,Ahmedabad,Gujarat,2024-02-20,FALSE,1002,102,2024-08-03,Delivered,799
3,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,TRUE,1003,103,2024-08-05,Shipped,7498
4,103,Rohan,Gupta,rohan.g@email.com,Delhi,Delhi,2024-03-10,TRUE,1008,103,2024-08-20,Delivered,899
5,104,Sneha,Reddy,sneha.r@email.com,Hyderabad,Telangana,2024-04-05,FALSE,1005,104,2024-08-12,Cancelled,2999
6,105,Vikram,Singh,vikram.s@email.com,Jaipur,Rajasthan,2024-05-12,TRUE,1006,105,2024-08-15,Delivered,5898
7,106,Ananya,Iyer,ananya.i@email.com,Chennai,Tamil Nadu,2024-06-18,FALSE,1007,106,2024-08-18,Pending,1299
8,107,Karan,Mehta,karan.m@email.com,Pune,Maharashtra,2024-07-22,TRUE,1009,107,2024-08-25,Shipped,6098
9,108,Divya,Nair,divya.n@email.com,Kochi,Kerala,2024-08-30,FALSE,1010,108,2024-08-28,Delivered,1598


Q21. Write a query using JOINs across three tables (orders → order_items → products) to show: order_id, product_name, quantity, unit_price, and discount_pct for each order item. 

In [24]:
pd.read_sql("SELECT o.order_id,p.product_name,oi.quantity,oi.unit_price,oi.discount_pct FROM orders o JOIN order_items oi ON o.order_id = oi.order_id JOIN products p ON oi.product_id = p.product_id",conn)


,order_id,product_name,quantity,unit_price,discount_pct
0,1001,Wireless Earbuds,2,1499,0
1,1001,Laptop Stand,1,899,10
2,1002,Cotton T-Shirt,1,799,0
3,1003,Smart Watch,1,2999,0
4,1003,Running Shoes,1,4599,5
5,1004,Bluetooth Speaker,1,3499,0
6,1005,Smart Watch,1,2999,0
7,1006,Wireless Earbuds,1,1499,10
8,1006,Running Shoes,1,4599,5
9,1007,Bedsheet Set,1,1299,0


Q22. Explain the difference between LEFT JOIN and RIGHT JOIN with an example from this schema. When would you use a FULL OUTER JOIN? 
 
 LEFT JOIN                                  | RIGHT JOIN                                
                                            |
 Keeps all rows from the left table         | Keeps all rows from the right table       
 Missing matches on the right become `NULL` | Missing matches on the left become `NULL` 



 A FULL OUTER JOIN is used when we want to join tables to-
All matching rows from both tables.
All unmatched rows from the left table.
All unmatched rows from the right table.


Q23. Identify all Foreign Key relationships in the schema. Explain what would happen if you tried to insert an order with customer_id = 999 (which doesn't exist in customers).
 

Foreign Key relationships -
orders.customer_id → customers.customer_id
order_items.order_id → orders.order_id
order_items.product_id → products.product_id

INSERT INTO orders (order_id, customer_id, order_date)
VALUES (101, 999, '2024-08-15');
If there is no customer with customer_id = 999 in the customers table, the database will reject the insert and raise a Foreign Key Constraint Violation error. This ensures that every order references a valid customer. Since customer 999 does not exist, inserting the order would break referential integrity.


Section E — Advanced Concepts (CASE, ACID, Transactions) 

Q24. Write a query using CASE to classify products into price tiers: 
  • 'Budget'    → unit_price < 1000 
  • 'Mid-Range' → unit_price BETWEEN 1000 AND 3000 
  • 'Premium'   → unit_price > 3000 
Display: product_name, unit_price, price_tier. 

In [25]:
pd.read_sql("""
SELECT
    product_name,
    unit_price,
    CASE
        WHEN unit_price < 1000 THEN 'Budget'
        WHEN unit_price BETWEEN 1000 AND 3000 THEN 'Mid-Range'
        ELSE 'Premium'
    END AS product_tier
FROM products
""",conn)


,product_name,unit_price,product_tier
0,Wireless Earbuds,1499,Mid-Range
1,Cotton T-Shirt,799,Budget
2,Smart Watch,2999,Mid-Range
3,Running Shoes,4599,Premium
4,Bluetooth Speaker,3499,Premium
5,Bedsheet Set,1299,Mid-Range
6,Laptop Stand,899,Budget
7,Cushion Covers(Set),599,Budget


Q25. Using a CASE statement inside an aggregate function, count how many orders are 'Delivered' vs 'Not Delivered' (all other statuses). Display the result in a single row. 

In [26]:
pd.read_sql("""
    SELECT
    SUM(CASE WHEN status = 'Delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN status <> 'Delivered' THEN 1 ELSE 0 END) AS not_delivered_orders
FROM orders;
""",conn)


,delivered_orders,not_delivered_orders
0,6,4


Q26. Explain each letter of ACID: 
  • A – Atomicity 
  • C – Consistency 
  • I – Isolation 
  • D – Durability 
Give a real-world example (e.g., bank transfer) showing why each property is important. 


A(Atomicity)- All Operations Succeeeds or none do 
eg - If the transactions in bank account happens it will be completedin one go or else it will be rejected(terminated)
     If ₹500 is transferred from Account A to Account B:
    Debit from A
    Credit to B
If the system fails after debiting A but before crediting B, the transaction is rolled back, and A still has the original amount.


C(Consistency)-The database remains must remain in a valid state before or after the transaction 
eg- If the amount in A's amount is 3000 and in B's account is 2000 ( before the transaction )
    If the A transfer 500 amount to B then the consistent state would be 
    in A's amount is 2500 and in B's account 2500(after the transaction)

I(Isolation)- Each transactions occur independently without interference 
eg-If two users try to withdraw from the same account at the same time:
Each transaction executes independently
One transaction does not see the intermediate state of another

D(Durability)-Once Comitted Changes are permanent even ,if system crashes
eg-If a transaction is successfully completed and committed:
Even if the server crashes immediately after
The updated balance remains saved when the system restarts

Q27. Write a SQL transaction that does the following atomically: 
  1. Insert a new order (order_id=1011, customer_id=102, today's date, 'Pending', 1598.00) 
  2. Insert two order items for that order 
  3. Update the stock_qty of the purchased products 
  4. If any step fails, ROLLBACK the entire transaction. Otherwise, COMMIT. 
Write the complete BEGIN...COMMIT/ROLLBACK block. 


try:
    cursor.execute("BEGIN")
    cursor.execute("""
        INSERT INTO orders VALUES(1011,102,DATE('now'),'Pending',1598.00)
    """)
    cursor.execute("""
        INSERT INTO order_items VALUES(5016,1011,201,2,500.00,0)
    """)
    cursor.execute("""
        INSERT INTO order_items VALUES(5017,1011,206,1,598.00,0)
    """)
    cursor.execute("UPDATE products SET stock_qty = stock_qty - 2 WHERE product_id = 201")
    cursor.execute("UPDATE products SET stock_qty = stock_qty - 1 WHERE product_id = 206")
    conn.commit()
    
except Exception as e:
    conn.rollback()
    print(f"Transaction rolled back: {e}")

## Summary

| Step | What was done |
|------|---------------|
| Schema design | 4 normalized tables with Foreign Key constraints and CHECK constraints |
| Indexes | 4 indexes created for query performance |
| Data insertion | Parameterized queries with INSERT OR IGNORE to prevent duplicates |
| Analysis |  SQL queries covering JOINs, aggregations,CASE |
| Verification | Automated integrity checks on all tables |

### Key Concepts Used
- `CREATE TABLE IF NOT EXISTS` — idempotent schema creation
- `INSERT OR IGNORE` — safe re-runs without duplicate errors  
- `executemany()` with parameterized `?` — 
- `FOREIGN KEY` constraints — referential integrity
- 'JOINS' -To extract information merging table
- 'CASE'-for multiple conditions
- Functions wrapping every step — reusable, testable code

In [27]:
conn.close()